In [1]:


import pandas as pd
import mysql.connector

# =========================
# DATABASE CONFIG
# =========================

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "actowiz",
    "database": "flipkart_minutes_final"
}

PINCODE_FILE = "ex/act-jnssav-7544-pincodes (1).xlsx"
PRODUCT_FILE = "ex/Flipkart Product vise locations.xlsx"
MAPPING_FILE = "ex/Mapping Sheet.xlsx"

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS pincodes (
    id INT AUTO_INCREMENT PRIMARY KEY,
    location VARCHAR(255),
    pincode VARCHAR(50),
    status VARCHAR(20) DEFAULT 'pending'
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INT AUTO_INCREMENT PRIMARY KEY,
               EAN_Code VARCHAR(50),
    city VARCHAR(255),
    product_title VARCHAR(500),
    flipkart_minutes_url TEXT
)
""")

pincode_df = pd.read_excel(PINCODE_FILE, sheet_name='Flipkart Minutes')

pincode_df.columns = pincode_df.columns.str.strip()
print(pincode_df.head())
print("Pincode Columns:")
print(pincode_df.columns.tolist())

insert_pincode_query = """
INSERT INTO pincodes (
    location,
    pincode,
    status
)
VALUES (%s, %s, %s)
"""

for _, row in pincode_df.iterrows():

    location = str(row["Location"]).strip()
    pincode = str(row["Pincode"]).strip()

    cursor.execute(
        insert_pincode_query,
        (location, pincode,'pending')
    )

conn.commit()

print("Pincode data inserted successfully.")

# =========================
# READ PRODUCT FILE
# =========================

product_df = pd.read_excel(PRODUCT_FILE)


product_df.columns = product_df.columns.str.strip()
print(product_df.head())
print("\nProduct Columns:")
print(product_df.columns.tolist())

mapping_df = pd.read_excel(MAPPING_FILE,skiprows=1,sheet_name="Flipkart Minutes and Grocery", dtype={"EAN Code": str})
# print(mapping_df['Unnamed: 1'])
mapping_df.columns = mapping_df.columns.str.strip()
print(mapping_df.head())
print("\nMapping Columns:")
print(mapping_df.columns.tolist())
print(mapping_df['Unnamed: 2'].head())

mapping_df = mapping_df[
    ["EAN Code","Article Description", "Unnamed: 2"]
]
print(mapping_df.head())



     Location  Pincode
0  Chandigarh   160101
1  Chandigarh   160102
2  Chandigarh   160014
3  Chandigarh   160047
4  Chandigarh   160003
Pincode Columns:
['Location', 'Pincode']
Pincode data inserted successfully.
                  product_title        City
0  Bhujialalji Aloo Bhujia 150g  Chandigarh
1  Bhujialalji Aloo Bhujia 150g    Dehradun
2  Bhujialalji Aloo Bhujia 150g     Lucknow
3  Bhujialalji Aloo Bhujia 150g       Delhi
4  Bhujialalji Aloo Bhujia 150g      Jaipur

Product Columns:
['product_title', 'City']
        EAN Code                 Article Description  \
0  8908017753082         Bhujialalji Soya Stick 150g   
1  8908017753051  Bhujialalji Mumbaiya Bhelpuri 150g   
2  8908017753013    Bhujialalji Bikaneri Bhujia 150g   
3  8908017753129        Bhujialalji Mota Bhujia 150g   
4  8908017753105    Bhujialalji Bikaneri Bhujia 400g   

                                          Unnamed: 2  \
0  https://www.flipkart.com/bikano-punjabi-masala...   
1  https://www.flipkart.com/

In [3]:

mapping_df.rename(
    columns={
        "Article Description": "product_title",
        "Unnamed: 2": "flipkart_minutes_url"
    },
    inplace=True
)
mapping_df

,EAN Code,product_title,flipkart_minutes_url
0,8908017753082,Bhujialalji Soya Stick 150g,https://www.flipkart.com/bikano-punjabi-masala...
1,8908017753051,Bhujialalji Mumbaiya Bhelpuri 150g,https://www.flipkart.com/bikano-punjabi-masala...
2,8908017753013,Bhujialalji Bikaneri Bhujia 150g,https://www.flipkart.com/bikano-bikaneri-bhuji...
3,8908017753129,Bhujialalji Mota Bhujia 150g,https://www.flipkart.com/bhujialalji-mota-bhuj...
4,8908017753105,Bhujialalji Bikaneri Bhujia 400g,https://www.flipkart.com/bhujialalji-bikaneri-...
5,8908017753099,Bhujialalji Aloo Bhujia 400g,https://www.flipkart.com/baker-s-dozen-high-pr...
6,8908017753136,Bhujialalji Navratna Mix 400g,https://www.flipkart.com/bikano-khatta-meetha/...
7,8908017753112,Bhujialalji Khatta Meetha 400g,https://www.flipkart.com/bikano-khatta-meetha/...
8,8908017753006,Bhujialalji Bikaneri Bhujia 1kg,https://www.flipkart.com/bikaji-bhujia-no-1/p/...
9,8908017753181,Bhujialalji Aloo Bhujia 1kg,https://www.flipkart.com/baker-s-dozen-high-pr...


In [6]:
mapping_df = mapping_df[
    mapping_df["flipkart_minutes_url"].notna() &
    (mapping_df["flipkart_minutes_url"].astype(str).str.strip() != "") &
    (mapping_df["flipkart_minutes_url"].astype(str).str.upper() != "N/A")
]
mapping_df

,EAN Code,product_title,flipkart_minutes_url
0,8908017753082,Bhujialalji Soya Stick 150g,https://www.flipkart.com/bikano-punjabi-masala...
1,8908017753051,Bhujialalji Mumbaiya Bhelpuri 150g,https://www.flipkart.com/bikano-punjabi-masala...
2,8908017753013,Bhujialalji Bikaneri Bhujia 150g,https://www.flipkart.com/bikano-bikaneri-bhuji...
3,8908017753129,Bhujialalji Mota Bhujia 150g,https://www.flipkart.com/bhujialalji-mota-bhuj...
4,8908017753105,Bhujialalji Bikaneri Bhujia 400g,https://www.flipkart.com/bhujialalji-bikaneri-...
5,8908017753099,Bhujialalji Aloo Bhujia 400g,https://www.flipkart.com/baker-s-dozen-high-pr...
6,8908017753136,Bhujialalji Navratna Mix 400g,https://www.flipkart.com/bikano-khatta-meetha/...
7,8908017753112,Bhujialalji Khatta Meetha 400g,https://www.flipkart.com/bikano-khatta-meetha/...
8,8908017753006,Bhujialalji Bikaneri Bhujia 1kg,https://www.flipkart.com/bikaji-bhujia-no-1/p/...
9,8908017753181,Bhujialalji Aloo Bhujia 1kg,https://www.flipkart.com/baker-s-dozen-high-pr...


In [8]:

print("*"*20)
print(len(mapping_df['flipkart_minutes_url'].unique()))
print("\nFiltered Mapping Data:")
print(mapping_df.head())

mapping_df = mapping_df[
    ["EAN Code","product_title", "flipkart_minutes_url"]
]

********************
18

Filtered Mapping Data:
        EAN Code                       product_title  \
0  8908017753082         Bhujialalji Soya Stick 150g   
1  8908017753051  Bhujialalji Mumbaiya Bhelpuri 150g   
2  8908017753013    Bhujialalji Bikaneri Bhujia 150g   
3  8908017753129        Bhujialalji Mota Bhujia 150g   
4  8908017753105    Bhujialalji Bikaneri Bhujia 400g   

                                flipkart_minutes_url  
0  https://www.flipkart.com/bikano-punjabi-masala...  
1  https://www.flipkart.com/bikano-punjabi-masala...  
2  https://www.flipkart.com/bikano-bikaneri-bhuji...  
3  https://www.flipkart.com/bhujialalji-mota-bhuj...  
4  https://www.flipkart.com/bhujialalji-bikaneri-...  


In [ ]:
sorted(product_df['product_title'].unique())
#  'Bhujialalji Bikaneri Bhujia,Rajasthani Bhujia, Namkeen Tea Time Fresh & Healthy Snack 1kg',
#  'Bhujialalji Khatta Meetha Namkeen Fresh Healthy Snack Mixture 1kg',
#  'Bhujialalji Navratan Mix 150g',
#  'Bhujialalji Navratna Mix Namkeen Fresh Healthy Snack Mixture 1kg',


['Bhujialalji Aloo Bhujia 150g',
 'Bhujialalji Aloo Bhujia 400g',
 'Bhujialalji Aloo Bhujia Namkeen Crsipy & Light snacks 1kg',
 'Bhujialalji Bikaneri Bhujia 150g',
 'Bhujialalji Bikaneri Bhujia 400g',
 'Bhujialalji Bikaneri Bhujia,Rajasthani Bhujia, Namkeen Tea Time Fresh & Healthy Snack 1kg',
 'Bhujialalji Khatta Meetha 150g',
 'Bhujialalji Khatta Meetha 400g',
 'Bhujialalji Khatta Meetha Namkeen Fresh Healthy Snack Mixture 1kg',
 'Bhujialalji Mota Bhujia 150g',
 'Bhujialalji Mumbaiya Bhelpuri 150g',
 'Bhujialalji Navratan Mix 150g',
 'Bhujialalji Navratna Mix 400g',
 'Bhujialalji Navratna Mix Namkeen Fresh Healthy Snack Mixture 1kg',
 'Bhujialalji Soya Stick 150g']

In [30]:
temp=pd.merge(
    mapping_df,
    product_df,
    on="product_title",
    how="left" 
)


In [45]:
unique_products = temp.drop_duplicates(subset=["product_title"])

unique_products.sort_values(by="product_title")

,EAN Code,product_title,flipkart_minutes_url,City
75,NaN,Bhujialalji Aloo Bhujia 150g,https://www.flipkart.com/rozana-papad-moong-pu...,Chandigarh
69,8908017753181,Bhujialalji Aloo Bhujia 1kg,https://www.flipkart.com/baker-s-dozen-high-pr...,NaN
42,8908017753099,Bhujialalji Aloo Bhujia 400g,https://www.flipkart.com/baker-s-dozen-high-pr...,Jaipur
85,,Bhujialalji Aloo Bhujia Namkeen Crsipy & Light...,https://www.flipkart.com/rozana-papad-moong-pu...,Bangalore
14,8908017753013,Bhujialalji Bikaneri Bhujia 150g,https://www.flipkart.com/bikano-bikaneri-bhuji...,Jaipur
68,8908017753006,Bhujialalji Bikaneri Bhujia 1kg,https://www.flipkart.com/bikaji-bhujia-no-1/p/...,NaN
28,8908017753105,Bhujialalji Bikaneri Bhujia 400g,https://www.flipkart.com/bhujialalji-bikaneri-...,Vijayawada
80,NaN,Bhujialalji Khatta Meetha 150g,https://www.flipkart.com/rozana-papad-moong-pu...,Jaipur
71,8908017753204,Bhujialalji Khatta Meetha 1kg,https://www.flipkart.com/bikaji-tana-tan/p/itm...,NaN
59,8908017753112,Bhujialalji Khatta Meetha 400g,https://www.flipkart.com/bikano-khatta-meetha/...,Lucknow


In [21]:
product_title_p=product_df['product_title'].unique()
mapping_title_p=mapping_df['product_title'].unique()

In [29]:
print(mapping_title_p)
print(product_title_p)

<StringArray>
[                              'Bhujialalji Soya Stick 150g',
                        'Bhujialalji Mumbaiya Bhelpuri 150g',
                          'Bhujialalji Bikaneri Bhujia 150g',
                              'Bhujialalji Mota Bhujia 150g',
                          'Bhujialalji Bikaneri Bhujia 400g',
                              'Bhujialalji Aloo Bhujia 400g',
                             'Bhujialalji Navratna Mix 400g',
                            'Bhujialalji Khatta Meetha 400g',
                           'Bhujialalji Bikaneri Bhujia 1kg',
                               'Bhujialalji Aloo Bhujia 1kg',
                              'Bhujialalji Navratna Mix 1kg',
                             'Bhujialalji Khatta Meetha 1kg',
                           'ROZANA URAD SPECIAL PAPAD 400 G',
                          'ROZANA MOONG SPECIAL PAPAD 400 G',
                          'ROZANA PAPAD MOONG PUNJABI 400 G',
                              'Bhujialalji Aloo Bhujia 1

In [28]:
mapping_df_temp = pd.DataFrame({
    "product_title": mapping_title_p
})

product_df_temp = pd.DataFrame({
    "product_title": product_title_p
})

test = pd.merge(
    mapping_df_temp,
    product_df_temp,
    on="product_title",
    how="inner"
)

print(len(test["product_title"].unique()))

11


In [12]:
temp['flipkart_minutes_url'].unique()

<StringArray>
[                                                     'https://www.flipkart.com/rozana-papad-moong-punjabi-masala/p/itm2734186c3775a?pid=SNSGG8RP4VN5HWQS&lid=LSTFRYHFQ2VBRAFRGM5PFHUUD&marketplace=HYPERLOCAL&pageUID=1779697447371',
 'https://www.flipkart.com/baker-s-dozen-high-protein-chips-masala-mania-zero-maida-baked/p/itm44906a975b4b0?pid=SNSGG8RPF6G8ZEGK&lid=LSTCHPHHDYX5BHWAXFHCDJVHL&marketplace=HYPERLOCAL&shopId=ahm_115_wh_hl_01&pageUID=1778751805733',
                                                      'https://www.flipkart.com/rozana-papad-moong-punjabi-masala/p/itm2734186c3775a?pid=SNSGG8RPF96M9YPZ&lid=LSTFRYHFQ2VBRAFRGM5PFHUUD&marketplace=HYPERLOCAL&pageUID=1779697535730',
                    'https://www.flipkart.com/bhujialalji-mota-bhujia/p/itmb3c7559585928?pid=SNSGG8RPHYGHH8C6&lid=LSTSNSGG8RPHYGHH8C6M1DPVS&marketplace=HYPERLOCAL&cmpid=content_snack-savourie_8965229628_gmc&pageUID=1778751569272',
                                    'https://www.flipkart.com/

In [19]:
merged_df_temp = pd.merge(
    product_df,
    mapping_df,
    on="product_title",
    how="left",
    indicator=True
)

# rows that did NOT match
not_matched = merged_df_temp [merged_df_temp["_merge"] == "left_only"]

print(not_matched["product_title"].unique())

<StringArray>
[                                                            'Bhujialalji Navratan Mix 150g',
 'Bhujialalji Bikaneri Bhujia,Rajasthani Bhujia, Namkeen Tea Time Fresh & Healthy Snack 1kg',
                         'Bhujialalji Khatta Meetha Namkeen Fresh Healthy Snack Mixture 1kg',
                          'Bhujialalji Navratna Mix Namkeen Fresh Healthy Snack Mixture 1kg']
Length: 4, dtype: str


In [ ]:

merged_df = pd.merge(
    product_df,
    mapping_df,
    on="product_title",
    how="inner"   # only matched products with URL
)

print("\nMerged Data:")
print(merged_df.head())

# =========================
# INSERT PRODUCTS
# =========================

insert_product_query = """
INSERT INTO products (
    Ean_Code,
    city,
    product_title,
    flipkart_minutes_url
)
VALUES (%s, %s, %s, %s)
"""

for _, row in merged_df.iterrows():

    ean_code = str(row["EAN Code"]).strip()
    city = str(row["City"]).strip()
    product_title = str(row["product_title"]).strip()
    url = str(row["flipkart_minutes_url"]).strip()

    cursor.execute(
        insert_product_query,
        (
            ean_code,
            city,
            product_title,
            url
        )
    )

conn.commit()

print("Product data inserted successfully.")
